# Najafi et al. (2020) on DANDI:000016, notebook 8: circuit models

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here (the full table is in notebook 1):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `fig5a-nonselective-model` | Non-selective model: I below E | 08 | tested (simulation) |
| `fig5bc-selective-models-match` | Selective models: E and I match near ratio 1 | 08 | tested (simulation) |
| `figs9-signal-selective-best` | Signal-selective model is most accurate | 08 | tested (simulation) |

These are simulations of the linearised rate model of the STAR Methods (model.py); they use no data
from the dandiset. Two details of the Methods admit more than one reading, and all four
combinations are run (model.py): the sign of the printed inhibitory weights, which are negative
while the equations already subtract inhibition ("signed": the printed values put into the
equations; "magnitude": their absolute values), and the connection scale, which the extracted text
gives as w/(cN) in a garbled formula that could be w/sqrt(cN) ("cN", "sqrt_cN"). A question is
asked about both. Per the claims file, when readings give different outcomes the claim is blocked as
methods_underspecified. The ratio of E-to-I to I-to-E strength runs from 0.1 to 10
on a log scale (claims file; 21 points). Non-selective and SNR-selective points use one network and
one pick of 50 read-out neurons with the input-averaged S/N (Methods); signal-selective points
average 25 networks and 100 picks.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import model as M
import najafi as nj   # names the dataset; the model uses none of its data

pd.set_option("display.width", 200)
print("Parameters:", M.P)
print("Ratios:", np.round(M.RATIOS, 3).tolist())
RES = {}
READINGS = [(sc, sg) for sc in ("cN", "sqrt_cN") for sg in ("signed", "magnitude")]
for k_r, reading in enumerate(READINGS):
    scale, sign = reading
    rng = np.random.default_rng([nj.SEED, 50, k_r])
    out = {}
    net = M.network(rng, "nonselective", sign, scale)
    pE = rng.choice(M.P["NE"], (1, M.P["n"]), replace=False); pI = rng.choice(M.P["NI"], (1, M.P["n"]), replace=False)
    st = M.readout_stats(net, pE, pI)
    pE20 = np.stack([rng.choice(M.P["NE"], M.P["n"], replace=False) for _ in range(20)])
    pI20 = np.stack([rng.choice(M.P["NI"], M.P["n"], replace=False) for _ in range(20)])
    st20 = M.readout_stats(net, pE20, pI20)
    out["nonselective"] = {eps: M.accuracy(st, M.RATIOS, averaged=(eps == 0), eps=eps) for eps in (0, 0.25, 0.5)}
    out["nonselective_20picks"] = M.accuracy(st20, M.RATIOS)
    # signal-selective: 25 networks, 100 picks each
    acc_s = {eps: {"E": [], "I": []} for eps in (0, 0.25, 0.5)}
    for k in range(25):
        nets = M.network(rng, "signal", sign, scale)
        pe = np.stack([rng.choice(M.P["NE"], M.P["n"], replace=False) for _ in range(100)])
        pi = np.stack([rng.choice(M.P["NI"], M.P["n"], replace=False) for _ in range(100)])
        sts = M.readout_stats(nets, pe, pi)
        for eps in acc_s:
            a = M.accuracy(sts, M.RATIOS, averaged=False, eps=eps)
            acc_s[eps]["E"].append(a["E"]); acc_s[eps]["I"].append(a["I"])
    out["signal"] = {eps: {p: np.mean(v[p], 0) for p in ("E", "I")} for eps, v in acc_s.items()}
    out["signal_network_sd"] = {p: np.std(acc_s[0][p], 0) for p in ("E", "I")}
    # SNR-selective: reweighted at each ratio from the non-selective network
    acc_n = {eps: {"E": [], "I": []} for eps in (0, 0.25, 0.5)}
    bp = M._parts(net)
    for r in M.RATIOS:
        ns = M.snr_selective(net, np.sqrt(r), bp)
        stn = M.readout_stats(ns, pE, pI)
        for eps in acc_n:
            a = M.accuracy(stn, np.array([r]), averaged=(eps == 0), eps=eps)
            acc_n[eps]["E"].append(a["E"][0]); acc_n[eps]["I"].append(a["I"][0])
    out["snr"] = {eps: {p: np.array(v[p]) for p in ("E", "I")} for eps, v in acc_n.items()}
    RES[reading] = out
    print(reading, "done", flush=True)

In [ ]:
for reading in RES:
    for model in ("nonselective", "signal", "snr"):
        a = RES[reading][model][0]
        print(f"{str(reading):23s} {model:12s} E: " + " ".join(f"{v:.3f}" for v in a["E"]))
        print(f"{'':23s} {'':12s} I: " + " ".join(f"{v:.3f}" for v in a["I"]))
    print(f"{str(reading):23s} non-selective, mean of 20 picks, E {RES[reading]['nonselective_20picks']['E'][[0, 10, 20]].round(3).tolist()}, "
          f"I {RES[reading]['nonselective_20picks']['I'][[0, 10, 20]].round(3).tolist()} (ratios 0.1, 1, 10)")
    print(f"{str(reading):23s} signal-selective SD over 25 networks at ratio 1: E {RES[reading]['signal_network_sd']['E'][10]:.4f}, I {RES[reading]['signal_network_sd']['I'][10]:.4f}")
print("Accuracies lie between %.3f and %.3f" % (min(min(v[0][p].min() for p in 'EI') for o in RES.values() for k, v in o.items() if k in ('nonselective', 'signal', 'snr')),
                                                max(max(v[0][p].max() for p in 'EI') for o in RES.values() for k, v in o.items() if k in ('nonselective', 'signal', 'snr'))))

## fig5a-nonselective-model

> In a model with non-selective inhibition, inhibitory neurons decode the choice less accurately
> than excitatory neurons for every ratio of excitatory-to-inhibitory to inhibitory-to-excitatory
> connection strength.

Reproduced when inhibitory accuracy is below excitatory at every value.

In [ ]:
for reading in RES:
    a = RES[reading]["nonselective"][0]
    print(f"{reading}: I below E at {int((a['I'] < a['E']).sum())} of {len(M.RATIOS)} ratios; smallest E - I {np.min(a['E'] - a['I']):.3f} "
          f"at ratio {M.RATIOS[np.argmin(a['E'] - a['I'])]:.2f}")

**Result: (to be written)**

## fig5bc-selective-models-match

> In signal-selective and SNR-selective models, inhibitory and excitatory neurons have matched
> decoding accuracy near the biologically plausible regime of equal excitatory-to-inhibitory and
> inhibitory-to-excitatory strengths; in all three models, accuracy rises faster for inhibitory than
> for excitatory neurons as the excitatory-to-inhibitory strength increases.

Reproduced when, for each selective model, the excitatory and inhibitory curves cross within a
factor of 2 of a ratio of 1 (0.5 to 2), and inhibitory accuracy rises faster than excitatory with the
ratio in all three models (here: the rise from the smallest to the largest ratio).

In [ ]:
for reading in RES:
    for model in ("nonselective", "signal", "snr"):
        a = RES[reading][model][0]
        d = a["I"] - a["E"]
        cross = [float(np.sqrt(M.RATIOS[k] * M.RATIOS[k + 1])) for k in range(len(d) - 1) if np.sign(d[k]) != np.sign(d[k + 1])]
        rise_E, rise_I = a["E"][-1] - a["E"][0], a["I"][-1] - a["I"][0]
        near = [c for c in cross if 0.5 <= c <= 2]
        print(f"{str(reading):23s} {model:12s} I - E from {d.min():+.3f} to {d.max():+.3f}; crossings at ratio {np.round(cross, 2).tolist()}; "
              f"within 0.5-2: {bool(near)}; rise E {rise_E:.3f}, I {rise_I:.3f}, I faster {rise_I > rise_E}")

**Result: (to be written)**

## figs9-signal-selective-best

> The signal-selective model has the highest decoding accuracy of the three models at every
> connection-strength ratio and level of differential correlations, in both cell types.

Differential correlations are added as Sigma_EE -> Sigma_EE + eps dh dh^T, eps = 0, 0.25, 0.5 (the
Figure S8 legend names Sigma_EE; the reading that also adds them to Sigma_II has no input difference
for the inhibitory population to use, so only Sigma_EE is run). With eps > 0 the S/N is that of the
network's own input difference for all models. Reproduced when the signal-selective accuracy is
highest at every ratio, eps and cell type.

In [ ]:
for reading in RES:
    for eps in (0, 0.25, 0.5):
        for pop in ("E", "I"):
            s_, n_, r_ = (RES[reading][k][eps][pop] for k in ("signal", "nonselective", "snr"))
            best = (s_ > n_) & (s_ > r_)
            print(f"{str(reading):23s} eps {eps:<4} {pop}: signal-selective highest at {int(best.sum())} of {len(best)} ratios; "
                  f"smallest margin {np.min(s_ - np.maximum(n_, r_)):+.4f}")

**Result: (to be written)**

## Summary

(to be written)